# Capítulo 3 - Cuaderno extra

Una persona misteriosa nos quiere contratar para crear una versión online del Juego del Calamar, quiere que recrees el juego de la galleta de azúcar de forma que puedas romper la galleta haciendo click y arrastrando en el contorno de la galleta.
<br>

<div style="display:flex; justify-content:center">
	<img src= "./images/juegoDelCalamar.png" style="width:300px">
</div>

<br>

Para ello, vamos a aplicar el detector de **Harris** que hemos visto en el **capítulo 4** para encontrar los puntos clave de cada galleta por los que deberá pasar cada jugador para que se rompa correctamente. 

Las galletas que vamos a tener en nuestro juego son:

<div style="display:flex; justify-content:center; gap:20px;">
	<img src= "./images/galletaCirculo.png" style="width:120px">
	<img src= "./images/galletaEstrella.png" style="width:120px">
	<img src= "./images/galletaTriangulo.png" style="width:120px">
	<img src= "./images/galletaParaguas.png" style="width:120px">
</div>

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (20.0, 20.0)
images_path = './images/'

## Implementación del detector Harris

Vamos a aplicar el **detector Harris** a cada una de las imágenes para poder encontrar cada uno de sus puntos claves. Luego le haremos **supresión de no-máximo** para quitarnos los puntos irrelevantes y vamos a comprobar si podemos realizar esta tarea con lo que llevamos dado en las prácticas.

Comenzamos creando algunas funciones auxiliares que nos vendrán bien en la ejecución de nuestro programa:

In [ ]:
from scipy import signal
def nonmaxsuppts(cim, radius, thresh):
    """ Binarize and apply non-maximum suppresion.   
    
        Args:
            cim: the harris 'R' image
            radius: the aperture size of local maxima window
            thresh: the threshold value for binarization
                    
        Returns: 
            r, c: two numpy vectors being the row (r) and the column (c) of each keypoint
    """   
    
    rows, cols = np.shape(cim)
    sze = 2 * radius + 1
    mx = signal.order_filter(cim, np.ones([sze, sze]), sze ** 2 - 1)
    bordermask = np.zeros([rows, cols]);
    bordermask[radius:(rows - radius), radius:(cols - radius)] = 1
    cim = np.array(cim)
    r, c = np.where((cim == mx) & (cim > thresh) & (bordermask == 1))
    return r, c

def gaussian_smoothing(image, sigma, w_kernel):
    """ Blur and normalize input image.   
    
        Args:
            image: Input image to be binarized
            sigma: Standard deviation of the Gaussian distribution
            w_kernel: Kernel aperture size
                    
        Returns: 
            smoothed_norm: Blurred image
    """   
    # Define 1D kernel
    s=sigma
    w=w_kernel
    kernel_1D = np.array([1 / (s*np.sqrt(2*np.pi)) * np.exp(-(z**2)/(2*s**2)) for z in range(-w,w+1)])
    
    # Apply distributive property of convolution
    vertical_kernel = kernel_1D.reshape(2*w+1,1)
    horizontal_kernel = kernel_1D.reshape(1,2*w+1)   
    gaussian_kernel_2D = signal.convolve2d(horizontal_kernel, vertical_kernel)   
    
    # NO hace falta usar cv2.CV_16 porque el kernel es positivo y suma 1, no hay overflow
    smoothed_img = cv2.filter2D(image, -1, gaussian_kernel_2D)
    
    # Normalize to [0, 254] values
    smoothed_norm = cv2.normalize(smoothed_img, None, 0, 254, cv2.NORM_MINMAX)

    return smoothed_norm

Ahora aplicamos lo anteriormente mencionado a cada una de las imágenes:

In [ ]:
# Leemos las imagenes originales
circulo_orig = cv2.imread(images_path + 'galletaCirculo.png')
estrella_orig = cv2.imread(images_path + 'galletaEstrella.png')
triangulo_orig = cv2.imread(images_path + 'galletaTriangulo.png')
paraguas_orig = cv2.imread(images_path + 'galletaParaguas.png')

# Aplicamos filtro gaussiano donde sea necesario
circulo = circulo_orig
estrella = gaussian_smoothing(estrella_orig, 3, 7)
triangulo = gaussian_smoothing(triangulo_orig, 3, 7)
paraguas = paraguas_orig

# Convertimos las ORIGINALES a RGB para mostrar
circulo_color = cv2.cvtColor(circulo_orig, cv2.COLOR_BGR2RGB)
estrella_color = cv2.cvtColor(estrella_orig, cv2.COLOR_BGR2RGB)
triangulo_color = cv2.cvtColor(triangulo_orig, cv2.COLOR_BGR2RGB)
paraguas_color = cv2.cvtColor(paraguas_orig, cv2.COLOR_BGR2RGB)

# Convertimos las FILTRADAS a escala de grises para Harris
circulo_gris = cv2.cvtColor(circulo, cv2.COLOR_BGR2GRAY)
estrella_gris = cv2.cvtColor(estrella, cv2.COLOR_BGR2GRAY)
triangulo_gris = cv2.cvtColor(triangulo, cv2.COLOR_BGR2GRAY)
paraguas_gris = cv2.cvtColor(paraguas, cv2.COLOR_BGR2GRAY)

# Parámetros constantes
size_sobel = 3
const_k = 0.05
size_window = 7
radius_non_maxima = 20

# Lista de imágenes (grises filtradas para Harris, color original para mostrar)
imagenes_grises = [circulo_gris, estrella_gris, triangulo_gris, paraguas_gris]
imagenes_color = [circulo_color, estrella_color, triangulo_color, paraguas_color]
nombres = ['Círculo', 'Estrella', 'Triángulo', 'Paraguas']

plt.figure(figsize=(16, 16))

# Procesamos cada imagen
for i, (img_gris, img_color, nombre) in enumerate(zip(imagenes_grises, imagenes_color, nombres)):
    # Calculamos los keypoints aplicanto Harris y cogiendo los máximos locales
    harris = cv2.cornerHarris(img_gris, size_sobel, size_window, const_k)
    threshold = 0.1 * harris.max()
    
    # Aplicamos supresión de no-máximos para evitar puntos muy cercanos
    r, c = nonmaxsuppts(harris, radius_non_maxima, threshold)
    
    # Convertimos las coordenadas a una lista de cv2.KeyPoint
    kps = [cv2.KeyPoint(float(c[j]), float(r[j]), None) for j in range(len(r))]
    
    # Dibujamos los keypoints
    image_corners = np.copy(img_color)
    cv2.drawKeypoints(image_corners, kps, image_corners, (0, 255, 0))
    
    # Mostramos la imagen con los keypoints
    plt.subplot(2, 2, i+1)
    plt.title(nombre, fontsize=16)
    plt.imshow(image_corners)
    plt.axis('off')

plt.tight_layout()
plt.show()

<br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br>
## Conclusión

Como podemos observar y, validando lo visto en clase, Harris funciona muy bien para las esquinas pero hace una labor nefasta cuando le pasamos una imagen que no tiene bordes claros.

Si quisiesemos crear este juego, necesitariamos usar otro tipo de detector que se especialice en círculos.

<div style="display:flex; justify-content:center">
	<img src= "./images/calamar2.png" style="width:250px">
</div>